# Quantum hardware validation of the optimized QuantumXCT circuit

Run the final optimized circuit **once** on a real IBM device (or a device-noise dry run) and
show it is good enough to trust for the paper:

1. **KL divergence** of the optimized circuit vs the 0-gate baseline, in three settings —
   exact (noiseless reference), device-noise simulation, and the real device — with and without
   readout-error correction. The KL reduction should survive device noise.
2. **Device quality on the qubits and links actually used** — per-qubit readout error (and T1/T2)
   and per-link two-qubit (ECR/CX) gate error, plus the transpiled depth and two-qubit gate count.
   Low errors on the used region are what make the result trustworthy.

This notebook is self-contained: it loads the saved optimized result (`run.json`), reuses the
package's validated `assert_quality` pipeline (dynamical decoupling, gate/measurement twirling, and
a tensored readout calibration), and saves figures + a JSON report to `results/hardware/`.

Run from `QuantumXCT/python` in the `qiskit-env`. For the real device, save your IBM account first
(`QiskitRuntimeService.save_account(...)`), then set `SUBMIT_HARDWARE = True` below.

In [ ]:
# Run from the QuantumXCT/python root, wherever Jupyter was started, so the data
# are found and every output goes to python/results/.
import os, sys
_root = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(_root, "qxct")) and os.path.dirname(_root) != _root:
    _root = os.path.dirname(_root)
os.chdir(_root)
for _p in (_root, os.path.join(_root, "benchmarks", "hardware")):
    if _p not in sys.path:
        sys.path.insert(0, _p)
print("working directory:", _root)

In [ ]:
# ---- parameters (edit these) ------------------------------------------------
RUN_JSON = "results/ovarian_cocultured/run.json"  # saved optimized result (topology + angles)
SUBMIT_HARDWARE = False     # False = exact + device-noise dry run only; True = also run ONCE on a real IBM device
BACKEND = None              # real device name e.g. "ibm_brisbane"; None = least busy (used only if SUBMIT_HARDWARE)
FAKE = "FakeBrisbane"       # device-noise model for the dry run and for the layout/quality preview
SHOTS = 8000
OUT = "results/hardware"

In [ ]:
import os, sys, json, time, warnings
sys.path.insert(0, os.getcwd())
warnings.filterwarnings("ignore", category=DeprecationWarning)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from qxct import QuantumXct, RunConfig, MitigationConfig
from qxct.exact import ExactScorer
from qxct.hardware import build_optimized_circuit, _initialize_to_stateprep, resolve_backend
from qxct.circuits import _GLOBAL_SEED
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from analysis_tools.data import load_ovarian
import qiskit_ibm_runtime.fake_provider as fp
import qxct.plotting as pl

os.makedirs(OUT, exist_ok=True)

def backend_name(b):
    n = getattr(b, "name", str(b))
    return n() if callable(n) else n

## 1. Load the optimized circuit and its exact (noiseless) KL

In [ ]:
run = json.load(open(RUN_JSON))
topo = [tuple(x) for x in run["cnot_sequence"]]
angles = run["crx_angles"]
genes = run["gene_list"]

H = load_ovarian()
qx = QuantumXct(RunConfig(), gene_list=genes)        # norm='l1' (default), as in run.json
qx.encode(H["ct1_mo"], H["ct2_mo"], H["ct1_co"], H["ct2_co"])
sc = ExactScorer(qx.initial1, qx.initial2, qx.target1, qx.target2)

exact_base, exact_opt = sc.baseline(), sc.kl(topo, angles)
print("Optimized topology:", "  ".join(f"{genes[c]}->{genes[t]}" for c, t in topo))
print(f"Exact KL:  baseline {exact_base:.4f}  ->  optimized {exact_opt:.4f} "
      f"({100*(1-exact_opt/exact_base):.1f}% reduction, {len(topo)} gates)")

## 2. Transpile for the device and inspect the layout

Transpiling is local and free (no hardware call). It tells us which physical qubits and which
two-qubit links the circuit lands on, and the transpiled depth / two-qubit gate count — the circuit
cost that the device errors below act on.

In [ ]:
if SUBMIT_HARDWARE:
    device = resolve_backend(BACKEND) if BACKEND else None
    if device is None:
        from qiskit_ibm_runtime import QiskitRuntimeService
        device = QiskitRuntimeService().least_busy(operational=True, simulator=False)
    print("Validating against REAL device:", backend_name(device))
else:
    device = getattr(fp, FAKE)()
    print("Validating against device-noise model (dry run):", backend_name(device))
dev_name = backend_name(device)

circ = _initialize_to_stateprep(build_optimized_circuit(qx.circ1, qx.circ2, topo, angles))
pm = generate_preset_pass_manager(backend=device, optimization_level=3, seed_transpiler=_GLOBAL_SEED)
isa = pm.run(circ)

layout = list(isa.layout.final_index_layout())
twoq_names = [n for n in ("ecr", "cx", "cz", "rzz") if n in device.target.operation_names]
gname = twoq_names[0]
links = sorted({tuple(sorted(isa.find_bit(q).index for q in ci.qubits))
                for ci in isa.data
                if ci.operation.num_qubits == 2 and ci.operation.name in twoq_names})
n2q = sum(v for k, v in isa.count_ops().items() if k in twoq_names)

print(f"physical qubits used : {layout}")
print(f"native 2q gate       : {gname}")
print(f"transpiled depth     : {isa.depth()}")
print(f"2q gate count        : {n2q}")
print(f"2q links used        : {links}")

## 3. Device quality on the qubits and links actually used

Per-qubit **readout error** and coherence (T1/T2), and per-link **two-qubit gate error**, read
straight from the device calibration (`backend.target`). These are the numbers that justify the
claim that the device is good enough for this circuit.

In [ ]:
tgt = device.target

qrows = []
for q in layout:
    qp = tgt.qubit_properties[q]
    def _err(op):
        try: return tgt[op][(q,)].error
        except Exception: return np.nan
    qrows.append(dict(qubit=q, readout_error=_err("measure"), sx_error=_err("sx"),
                      T1_us=(qp.t1 * 1e6 if qp.t1 else np.nan),
                      T2_us=(qp.t2 * 1e6 if qp.t2 else np.nan)))
qubit_quality = pd.DataFrame(qrows)

lrows = []
for l in links:
    e = np.nan
    for pair in (l, (l[1], l[0])):
        if pair in tgt[gname]:
            e = tgt[gname][pair].error; break
    lrows.append(dict(link=f"{l[0]}-{l[1]}", two_q_error=e))
link_quality = pd.DataFrame(lrows)

print("Per-qubit quality (physical qubits used):")
print(qubit_quality.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("\nPer-link two-qubit gate error (links used):")
print(link_quality.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print(f"\nmax readout error {qubit_quality.readout_error.max():.4f} | "
      f"max {gname} error {link_quality.two_q_error.max():.4f} | "
      f"mean {gname} error {link_quality.two_q_error.mean():.4f}")

## 4. Run the circuit once (device noise, and the real device if enabled)

`MitigationConfig.all_on` turns on dynamical decoupling, gate + measurement twirling, and a tensored
readout calibration. We score the baseline (0 gates) and the optimized circuit so the KL **reduction**
is measured on the same device.

In [ ]:
mit = MitigationConfig.all_on(shots=SHOTS)

report = {
    "backend_validated": dev_name,
    "topology": [f"{genes[c]}->{genes[t]}" for c, t in topo],
    "angles": angles, "shots": SHOTS,
    "transpiled_depth": isa.depth(), "transpiled_2q_gates": int(n2q),
    "two_q_gate": gname, "physical_layout": layout,
    "qubit_quality": qubit_quality.to_dict("records"),
    "link_quality": link_quality.to_dict("records"),
    "exact": {"baseline_kl": exact_base, "final_kl": exact_opt,
              "reduction_pct": 100 * (1 - exact_opt / exact_base)},
}

def run_setting(run_backend, tag):
    out = {}
    for name, (t, th) in {"baseline": ([], []), "optimized": (topo, angles)}.items():
        qx.result.topology, qx.result.angles = t, th
        t0 = time.time()
        r = qx.assert_on_hardware(backend=run_backend, mitigation=mit)
        r["wall_s"] = time.time() - t0
        # keep everything except the raw counts and the (large) assignment matrices
        rc = r.get("kl_readout_corrected")
        if isinstance(rc, dict):
            rc = {k: v for k, v in rc.items() if k != "assignment_matrices"}
        out[name] = {k: v for k, v in r.items() if not k.startswith("counts_")}
        out[name]["kl_readout_corrected"] = rc
    b, f = out["baseline"]["kl_total"], out["optimized"]["kl_total"]
    out["reduction_pct_raw"] = 100 * (1 - f / b)
    rc_b = (out["baseline"].get("kl_readout_corrected") or {}).get("kl_total")
    rc_f = (out["optimized"].get("kl_readout_corrected") or {}).get("kl_total")
    if rc_b and rc_f:
        out["reduction_pct_readout_corrected"] = 100 * (1 - rc_f / rc_b)
    report[tag] = out
    msg = f"[{tag}] KL baseline {b:.4f} -> optimized {f:.4f}  ({out['reduction_pct_raw']:.1f}% raw"
    if "reduction_pct_readout_corrected" in out:
        msg += f", {out['reduction_pct_readout_corrected']:.1f}% readout-corrected"
    print(msg + ")")

# device-noise dry run (always)
from qiskit_aer import AerSimulator
fake = getattr(fp, FAKE)()
run_setting(AerSimulator.from_backend(fake, seed_simulator=42), f"fake_noisy_{FAKE}")

# real device, once, if enabled
if SUBMIT_HARDWARE:
    print("submitting to", dev_name, "...")
    run_setting(device, f"hardware_{dev_name}")

print(f"\n[exact] baseline {exact_base:.4f} -> optimized {exact_opt:.4f}")

## 5. Figures for the paper

In [ ]:
HW_TAGS = [k for k in report if k.startswith("fake_noisy_") or k.startswith("hardware_")]

# --- (a) KL: baseline vs optimized across settings (exact, noisy, [hardware], readout-corrected) ---
labels, base_kl, opt_kl = ["exact"], [exact_base], [exact_opt]
for tag in HW_TAGS:
    short = tag.replace("fake_noisy_", "noisy:").replace("hardware_", "hw:")
    labels.append(short)
    base_kl.append(report[tag]["baseline"]["kl_total"])
    opt_kl.append(report[tag]["optimized"]["kl_total"])
    rc = (report[tag]["optimized"].get("kl_readout_corrected") or {}).get("kl_total")
    if rc is not None:
        labels.append(short + "\n(readout-corr)")
        base_kl.append((report[tag]["baseline"].get("kl_readout_corrected") or {}).get("kl_total"))
        opt_kl.append(rc)

x = np.arange(len(labels)); w = 0.38
fig, ax = plt.subplots(figsize=(2.2 + 1.5 * len(labels), 4.3))
ax.bar(x - w / 2, base_kl, w, color=pl.C_NON, label="baseline (0 gates)")
ax.bar(x + w / 2, opt_kl, w, color=pl.C_QX, label="optimized")
for xi, (bk, ok) in enumerate(zip(base_kl, opt_kl)):
    ax.annotate(f"{bk:.3f}", (xi - w / 2, bk), textcoords="offset points", xytext=(0, 2), ha="center", fontsize=7)
    ax.annotate(f"{ok:.3f}", (xi + w / 2, ok), textcoords="offset points", xytext=(0, 2), ha="center", fontsize=7)
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel("Combined KL divergence")
ax.set_title("Optimized circuit vs baseline: KL survives device noise")
ax.legend(frameon=False); ax.yaxis.grid(True, ls=":", alpha=0.6); ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig(f"{OUT}/hw_kl_comparison.png", dpi=300, bbox_inches="tight")
fig.savefig(f"{OUT}/hw_kl_comparison.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# --- (b) per-qubit readout error on the used qubits ---
fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.bar([str(q) for q in qubit_quality.qubit], qubit_quality.readout_error * 100, color=pl.C_CT1)
ax.set_xlabel("physical qubit"); ax.set_ylabel("readout error (%)")
ax.set_title(f"Per-qubit readout error on {dev_name} (qubits used)")
ax.yaxis.grid(True, ls=":", alpha=0.6); ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig(f"{OUT}/hw_readout_error.png", dpi=300, bbox_inches="tight")
fig.savefig(f"{OUT}/hw_readout_error.pdf", bbox_inches="tight")
plt.show()

# --- (c) per-link two-qubit gate error on the used links ---
fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.bar(link_quality.link, link_quality.two_q_error * 100, color=pl.C_LR)
ax.set_xlabel(f"{gname} link (physical qubits)"); ax.set_ylabel(f"{gname} gate error (%)")
ax.set_title(f"Per-link two-qubit gate error on {dev_name} (links used)")
ax.yaxis.grid(True, ls=":", alpha=0.6); ax.set_axisbelow(True)
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
fig.tight_layout()
fig.savefig(f"{OUT}/hw_2q_error.png", dpi=300, bbox_inches="tight")
fig.savefig(f"{OUT}/hw_2q_error.pdf", bbox_inches="tight")
plt.show()

## 6. Save the report

In [ ]:
fn = f"{OUT}/hardware_validation_report.json"
json.dump(report, open(fn, "w"), indent=1, default=float)
print("saved", fn)
for f in ("hw_kl_comparison", "hw_readout_error", "hw_2q_error"):
    print("saved", f"{OUT}/{f}.png", "/", ".pdf")

## How to read this

- **KL reduction survives noise.** Compare the optimized bar to the baseline bar in each setting. The
  exact reduction is the ideal; the device-noise and (if run) real-device reductions should stay
  clearly positive, and the **readout-corrected** pair should recover most of the exact reduction.
  That is the evidence the optimized circuit is doing real work on hardware, not riding noise.
- **The device region is clean.** The per-qubit readout errors and per-link two-qubit gate errors are
  for the exact qubits and links the circuit uses. Low, single-digit-percent errors on a shallow
  circuit (small transpiled depth, few two-qubit gates) are what make the KL result trustworthy.
- **For the paper:** set `SUBMIT_HARDWARE = True` (and optionally `BACKEND`), run once, and quote the
  real-device raw and readout-corrected reductions, the depth, the two-qubit gate count, and the
  max/mean device errors. Everything is also in `results/hardware/hardware_validation_report.json`.